In [1]:
!source .venv/bin/activate

In [2]:
#!/usr/bin/env python3
"""
Generate teacher-model reasoning traces on SATBench for use as SFT
training data for a smaller student model.
 
  Teacher : Qwen/Qwen3.6-35B-A3B  (thinking-mode chat model, MoE)
  Dataset : satbench_with_certificates_full.jsonl
            (SATBench rows enriched with Z3-verified certificates)
  Output  : one JSON file per datapoint inside --output_dir,
            containing the rendered prompt and the full teacher
            response (including the <think> ... </think> trace).
 
This version is aligned with the SATBench paper (arXiv:2505.14615v2):
  * Variable IDs are converted from 1-indexed DIMACS form to the
    structural form x(i,), x(i, j), x(i, j, k) used by the dataset's
    `readable`, `conditions`, and `variable_mapping` fields, derived
    from `dims`.
  * The user prompt notes explicitly that the order of the formal
    `clauses` list does NOT match the order of natural-language
    `conditions`; conditions follow `readable`.
  * The system prompt mirrors the SATBench evaluation prompt
    (Figure A3): constraints come only from <conditions>; the
    scenario adds no rules; variables are independent; unmentioned
    variables are irrelevant.
  * The teacher is asked to end with [SAT] / [UNSAT] and a structured
    Assignment: [...] array matching `dims` (Figure A4 format).
 
Typical use (single A100/H100 with bf16):
 
  python generate_teacher_traces.py \\
      --dataset satbench_with_certificates_full.jsonl \\
      --output_dir teacher_traces/ \\
      --start_idx 0 --end_idx 2100
 
The script is resumable: if a target output JSON already exists and
contains a non-empty "response" field, that index is skipped.
"""
 
from __future__ import annotations
 
import argparse
import json
import os
import re
import sys
import time
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple
 
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

/scratch/network/yd1202/COS598B-project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!nvidia-smi

Fri May  1 16:19:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:0D:00.0 Off |                    0 |
| N/A   59C    P0            214W /  300W |    4924MiB /  81920MiB |     99%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [18]:
messages = [
    {"role": "user", "content": "Type \"I love Qwen3.6\" backwards"},
]

chat_response = client.chat.completions.create(
    model="Qwen/Qwen3.6-35B-A3B",
    messages=messages,
    max_tokens=81920,
    temperature=1.0,
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
        # "top_k": 20,
    }, 
)
print("Chat response:", chat_response)


Chat response: ChatCompletion(id='4eee85c6-b1f6-4d79-a4b9-4b4d85227c39', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: `Type "I love Qwen3.6" backwards`\n   - Target string: `I love Qwen3.6`\n   - Task: Reverse the string exactly as given.\n\n2.  **Identify Key Components:**\n   - String to reverse: `I love Qwen3.6`\n   - Note: Includes spaces, punctuation (period), and numbers.\n   - Case sensitivity matters: `I` is uppercase, `Q` is uppercase.\n\n3.  **Perform Reversal (Mental/Step-by-Step):**\n   Original: `I love Qwen3.6`\n   Let\'s write it out backwards character by character:\n   - `6`\n   - `.`\n   - `3`\n   - `n`\n   - `e`\n   - `w`\n   - `Q`\n   - ` ` (space)\n   - `e`\n   - `v`\n   - `o`\n   - `l`\n   - ` ` (space)\n   - `I`\n\n   Combine: `6.3newQ evol I`\n\n   Wait, let\'s double-check:\n   Original: I (space) l o v e (space) Q w e n 3 . 6\n   

In [20]:
print("""
Chat response: ChatCompletion(id='4eee85c6-b1f6-4d79-a4b9-4b4d85227c39', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here\'s a thinking process:\n\n1.  **Analyze User Input:**\n   - User says: `Type "I love Qwen3.6" backwards`\n   - Target string: `I love Qwen3.6`\n   - Task: Reverse the string exactly as given.\n\n2.  **Identify Key Components:**\n   - String to reverse: `I love Qwen3.6`\n   - Note: Includes spaces, punctuation (period), and numbers.\n   - Case sensitivity matters: `I` is uppercase, `Q` is uppercase.\n\n3.  **Perform Reversal (Mental/Step-by-Step):**\n   Original: `I love Qwen3.6`\n   Let\'s write it out backwards character by character:\n   - `6`\n   - `.`\n   - `3`\n   - `n`\n   - `e`\n   - `w`\n   - `Q`\n   - ` ` (space)\n   - `e`\n   - `v`\n   - `o`\n   - `l`\n   - ` ` (space)\n   - `I`\n\n   Combine: `6.3newQ evol I`\n\n   Wait, let\'s double-check:\n   Original: I (space) l o v e (space) Q w e n 3 . 6\n   Reverse: 6 . 3 n e w Q (space) e v o l (space) I\n   Combined: `6.3newQ evol I`\n\n4.  **Verify:**\n   Let\'s count characters:\n   I (1) space (2) l (3) o (4) v (5) e (6) space (7) Q (8) w (9) e (10) n (11) 3 (12) . (13) 6 (14)\n   Reversed: 6 . 3 n e w Q   e v o l   I\n   Matches exactly.\n\n   Wait, is there any trick? The prompt says "Qwen3.6". I should just output the reversed string exactly as requested.\n   Let\'s carefully type it out: `6.3newQ evol I`\n\n   Check spacing: \n   Original: "I love Qwen3.6"\n   Backwards: "6.3newQ evol I"\n   Looks correct.\n\n5.  **Formulate Response:**\n   Just output the reversed string as requested. Keep it direct.\n   "6.3newQ evol I"\n\n   I\'ll double-check one more time:\n   I -> I\n    ->  \n   l -> l\n   o -> o\n   v -> v\n   e -> e\n    ->  \n   Q -> Q\n   w -> w\n   e -> e\n   n -> n\n   3 -> 3\n   . -> .\n   6 -> 6\n   Reverse order: 6 . 3 n e w Q [space] e v o l [space] I\n   Result: `6.3newQ evol I`\n\n   All good. Output matches.✅\n   Note: The prompt says "Qwen3.6", not "Qwen 3.6" or "Qwen 3.0". I\'ll stick exactly to the given text.\n   Final answer: `6.3newQ evol I`✅\n   I will output just that or with a brief confirmation if needed, but direct is best.\n   "6.3newQ evol I" is the exact reversal.\n   Proceed. \n   Self-Correction/Refinement during thought: \n   Wait, let\'s count carefully: \n   I(1) (2)l(3)o(4)v(5)e(6) (7)Q(8)w(9)e(10)n(11)3(12).(13)6(14)\n   Reverse: 6.(13)3(12)n(11)e(10)w(9)Q(8) (7)e(6)v(5)o(4)l(3) (2)I(1)\n   Result: 6.3newQ evol I\n   Perfect. No issues. Output matches.✅\n   I\'ll generate the response now. \n   Output: `6.3newQ evol I` (with or without quotes? The prompt says Type "..." backwards, so I\'ll just output the text itself, maybe in quotes if needed, but plain is fine. I\'ll just give the exact reversed string.)\n   I\'ll write: `6.3newQ evol I` \n   Done. \n   [Final Check] \n   Original: I love Qwen3.6\n   Reversed: 6.3newQ evol I\n   Matches. Output matches request.✅\n   Proceeds. \n   [Output Generation] -> `6.3newQ evol I` (I\'ll just output that directly) \n   Note: I should make sure capitalization is preserved exactly. "Q" stays "Q", "I" stays "I". Yes.\n   All good. \n   Response: `6.3newQ evol I`✅\n   (Self-Correction/Verification during process: I\'ll quickly type it out backwards to be 100% sure: 6 . 3 n e w Q [space] e v o l [space] I -> 6.3newQ evol I. Correct.) \n   Final Answer Generation. \n   [Output] 6.3newQ evol I\n   (I\'ll present it clearly) \n   "6.3newQ evol I" \n   Done. \n   [Proceeds]\n</think>\n\n6.3newQ evol I', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1776651514, model='Qwen/Qwen3.6-35B-A3B@main', object='chat.completion', service_tier=None, system_fingerprint=None, usage=CompletionUsage(completion_tokens=1255, prompt_tokens=21, total_tokens=1276, completion_tokens_details=None, prompt_tokens_details=None))
""")


Chat response: ChatCompletion(id='4eee85c6-b1f6-4d79-a4b9-4b4d85227c39', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Here's a thinking process:

1.  **Analyze User Input:**
   - User says: `Type "I love Qwen3.6" backwards`
   - Target string: `I love Qwen3.6`
   - Task: Reverse the string exactly as given.

2.  **Identify Key Components:**
   - String to reverse: `I love Qwen3.6`
   - Note: Includes spaces, punctuation (period), and numbers.
   - Case sensitivity matters: `I` is uppercase, `Q` is uppercase.

3.  **Perform Reversal (Mental/Step-by-Step):**
   Original: `I love Qwen3.6`
   Let's write it out backwards character by character:
   - `6`
   - `.`
   - `3`
   - `n`
   - `e`
   - `w`
   - `Q`
   - ` ` (space)
   - `e`
   - `v`
   - `o`
   - `l`
   - ` ` (space)
   - `I`

   Combine: `6.3newQ evol I`

   Wait, let's double-check:
   Original: I (space) l o v e (space) Q w e n 3 . 6
   Reverse: 6 . 3 n e w Q (space) e v o

In [4]:
# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
 
TEACHER_MODEL_ID = "Qwen/Qwen3.6-35B-A3B"
 
 
# Mirrors the spirit of the SATBench evaluation prompt (Figure A3) so the
# teacher reasons under the same guardrails as the student will be evaluated
# under. The "ground-truth certificate" framing is a distillation-only
# addition; the teacher must DERIVE the certificate, not just restate it.
SYSTEM_PROMPT = """You are an expert in propositional logic and Boolean
satisfiability (SAT). You are helping to generate high-quality reasoning
traces that a smaller student model will be fine-tuned on.
 
You will be given:
  * a natural-language logic puzzle (scenario + variable mapping +
    conditions + question), and
  * the underlying CNF formula (dims, num_vars, num_clauses, the raw
    DIMACS clauses, and the human-readable formula), and
  * a solver-verified Z3 certificate (a satisfying assignment for SAT
    instances, or an unsat core for UNSAT instances).
 
Reasoning rules (these match the SATBench evaluation protocol):
  * All constraints come ONLY from the <conditions> section. The
    <scenario> provides background and intuition, but does not impose
    any additional rules.
  * Variables represent INDEPENDENT decisions. Do not assume mutual
    exclusivity, totality, or any commonsense linkage unless the
    <conditions> state it explicitly.
  * Variables not mentioned in any condition are unconstrained and
    irrelevant to satisfiability; they may be set arbitrarily.
 
Task:
  1. Reason step by step. Map every natural-language condition to the
     underlying boolean variables, and either build a satisfying
     assignment by backtracking search or derive a contradiction by
     unit propagation.
  2. Use the provided Z3 certificate to anchor your reasoning, but you
     must SHOW the derivation. Do not simply restate the certificate.
  3. End your final answer with the SATBench evaluation format:
       - For SAT: a structured assignment matching `dims`, followed
         by the literal tag [SAT].
       - For UNSAT: a list of the conditions that together form the
         contradiction, followed by the literal tag [UNSAT].
 
Output structure (the <think> block becomes the training-time chain
of thought; the rest is the final answer):
 
<think>
... full step-by-step reasoning ...
</think>
 
Decision: <SAT or UNSAT>
 
Certificate:
  For SAT, provide a structured assignment in this exact form
  (a nested list whose shape equals `dims`, with 1 = True, 0 = False):
      Assignment: [[...], [...], ...]
  For UNSAT, list the natural-language condition numbers (1-indexed)
  that together cannot all be satisfied, e.g.:
      UNSAT core (condition numbers): [1, 3, 4]
 
Explanation:
  A short justification (2-5 sentences) that references the key
  conditions by their natural-language number.
 
[SAT] or [UNSAT]
"""
 
 
# ---------------------------------------------------------------------------
# DIMACS <-> structural variable conversion
# ---------------------------------------------------------------------------
 
def dimacs_to_struct(var_id: int, dims: Sequence[int]) -> str:
    """Convert a 1-indexed DIMACS variable id to the structural form
    x(i,) / x(i, j) / x(i, j, k) used by SATBench's `readable` field.
 
    Row-major (C-order) layout, matching the dataset.
    """
    if not dims:
        return f"x{var_id}"
    flat = var_id - 1
    indices: List[int] = []
    n = len(dims)
    for k in range(n):
        stride = 1
        for d in dims[k + 1:]:
            stride *= d
        indices.append(flat // stride)
        flat = flat % stride
    if n == 1:
        # SATBench writes 1-D variables as `x(0,)` (Python tuple style).
        return f"x({indices[0]},)"
    return f"x({', '.join(str(i) for i in indices)})"
 
 
def literal_to_struct(lit: int, dims: Sequence[int]) -> str:
    """Render a signed DIMACS literal in structural form, with negation."""
    sign = "¬" if lit < 0 else ""
    return f"{sign}{dimacs_to_struct(abs(lit), dims)}"
 
 
def clause_to_struct(clause: Sequence[int], dims: Sequence[int]) -> str:
    """Render an entire clause in structural form."""
    return "(" + " ∨ ".join(literal_to_struct(l, dims) for l in clause) + ")"
 
 
def reshape_assignment(assignment: Dict[Any, Any],
                       dims: Sequence[int]) -> Any:
    """Reshape a flat 1-indexed assignment dict into nested lists of 0/1
    matching `dims`. Missing keys default to 0 (False)."""
    if not dims:
        return None
    total = 1
    for d in dims:
        total *= d
 
    flat: List[int] = []
    for i in range(1, total + 1):
        v = assignment.get(str(i))
        if v is None:
            v = assignment.get(i, False)
        flat.append(1 if v else 0)
 
    def _reshape(values: List[int], shape: Sequence[int]) -> Any:
        if len(shape) == 1:
            return list(values[: shape[0]])
        d, rest = shape[0], shape[1:]
        chunk = 1
        for r in rest:
            chunk *= r
        return [_reshape(values[i * chunk:(i + 1) * chunk], rest)
                for i in range(d)]
 
    return _reshape(flat, list(dims))
 
 
# ---------------------------------------------------------------------------
# Certificate rendering
# ---------------------------------------------------------------------------
 
def render_sat_certificate(row: Dict[str, Any]) -> str:
    assignment = row.get("sat_assignment") or {}
    dims = row.get("dims") or []
    if not assignment:
        return "Certificate type: SAT (assignment is empty)"
 
    try:
        items = sorted(assignment.items(), key=lambda kv: int(kv[0]))
    except Exception:
        items = list(assignment.items())
 
    lines = []
    for var_id, val in items:
        try:
            struct = dimacs_to_struct(int(var_id), dims)
        except Exception:
            struct = f"x{var_id}"
        lines.append(f"  {struct} = {bool(val)}")
    flat_block = "\n".join(lines)
 
    structured = reshape_assignment(assignment, dims)
    structured_str = json.dumps(structured)
 
    sat_reason = row.get("sat_reason")
    extra = f"\nSolver note: {sat_reason}" if sat_reason else ""
 
    return (
        "Certificate type: SAT (satisfying assignment from Z3)\n"
        "Per-variable assignment (structural form):\n"
        f"{flat_block}\n"
        "Same assignment as a structured array matching `dims` "
        "(1 = True, 0 = False; this is the SATBench Assignment format):\n"
        f"  Assignment: {structured_str}{extra}"
    )
 
 
def render_unsat_certificate(row: Dict[str, Any]) -> str:
    core_indices: List[int] = row.get("unsat_core_clause_indices") or []
    clauses: List[List[int]] = row.get("clauses") or []
    dims = row.get("dims") or []
 
    lines = []
    for i in core_indices:
        if 0 <= i < len(clauses):
            cl = clauses[i]
            lines.append(
                f"  clauses[{i}] (DIMACS): {cl}"
                f"  ->  {clause_to_struct(cl, dims)}"
            )
        else:
            lines.append(f"  clauses[{i}]: <out of range>")
    core_block = "\n".join(lines) if lines else "  (empty)"
 
    unsat_reason = row.get("unsat_reason") or "(not provided)"
 
    return (
        "Certificate type: UNSAT (unsat core from Z3)\n"
        "UNSAT core clause indices (0-indexed into the formal `clauses` "
        f"list): {core_indices}\n"
        "These clauses, in structural form (match them to natural-language "
        "conditions by literal content, NOT by index — see ordering note "
        "below):\n"
        f"{core_block}\n"
        f"Solver-derived UNSAT reason: {unsat_reason}"
    )
 
 
def build_certificate_block(row: Dict[str, Any]) -> str:
    cert_type = row.get("certificate_type")
    if cert_type == "sat_assignment":
        return render_sat_certificate(row)
    if cert_type == "unsat_core":
        return render_unsat_certificate(row)
    return f"Certificate type: {cert_type!r} (no structured certificate)"
 
 
# ---------------------------------------------------------------------------
# Prompt assembly
# ---------------------------------------------------------------------------
 
def build_user_prompt(row: Dict[str, Any]) -> str:
    """Assemble the user-side prompt from one SATBench row + Z3 certificate.
 
    The prompt is split into:
      * "Puzzle" — exactly the fields a SATBench-evaluated student sees.
      * "Auxiliary information" — the formal CNF and the solver
        certificate, present here only to anchor the teacher's CoT.
    """
    conditions = row.get("conditions") or []
    if conditions:
        conditions_block = "\n".join(conditions)
    else:
        conditions_block = "(none)"
 
    sat_label = "SAT" if row.get("satisfiable") else "UNSAT"
 
    return (
        "## Puzzle (this is what the evaluated student model will see)\n\n"
        f"### Scenario\n{row.get('scenario', '')}\n\n"
        f"### Variable Mapping\n{row.get('variable_mapping', '')}\n\n"
        f"### Conditions (1-indexed; this is the canonical natural-language "
        f"order)\n{conditions_block}\n\n"
        f"### Question\n{row.get('question', '')}\n\n"
        "## Auxiliary information (NOT shown at evaluation time; use to "
        "ground your reasoning)\n\n"
        "### Underlying CNF formula\n"
        f"- dims: {row.get('dims', [])}\n"
        f"- num_vars: {row.get('num_vars', 0)}\n"
        f"- num_clauses: {row.get('num_clauses', 0)}\n"
        f"- clauses (DIMACS-style; 1-indexed flat variable IDs; negative "
        f"= negated literal): {row.get('clauses', [])}\n"
        f"- Readable CNF (this list IS in the same order as `Conditions` "
        f"above): {row.get('readable', '')}\n\n"
        "### Ordering note (important)\n"
        "The order of entries in `clauses` (the DIMACS list) does NOT "
        "in general match the order of `Conditions` / `Readable CNF`. "
        "When you cite a condition number in your final answer, use the "
        "1-indexed `Conditions` order (which matches `Readable CNF`). "
        "Match `clauses[i]` to a condition by comparing literal contents, "
        "not by index.\n\n"
        f"### Ground-truth satisfiability (from Z3)\n{sat_label}\n\n"
        "### Solver-verified certificate (DERIVE this, do not copy it)\n"
        f"{build_certificate_block(row)}\n\n"
        "Now solve the puzzle. Show the full chain of thought inside a "
        "single <think> ... </think> block, then output Decision / "
        "Certificate / Explanation in the structure given by the system "
        "prompt, ending with the literal tag [SAT] or [UNSAT]."
    )
 
 
# ---------------------------------------------------------------------------
# Response parsing
# ---------------------------------------------------------------------------
 
_THINK_RE = re.compile(r"<think>(.*?)</think>", flags=re.DOTALL)
 
 
def split_think(text: str) -> Tuple[Optional[str], str]:
    """Return (thinking_trace, final_answer)."""
    m = _THINK_RE.search(text)
    if m:
        thinking = m.group(1).strip()
        final = (text[:m.start()] + text[m.end():]).strip()
        return thinking, final
    if "</think>" in text:
        # Some chat templates pre-open <think>; only the closing tag appears
        # in the generated text.
        j = text.index("</think>")
        return text[:j].strip(), text[j + len("</think>"):].strip()
    return None, text.strip()
 
 
# ---------------------------------------------------------------------------
# I/O helpers
# ---------------------------------------------------------------------------
 
def load_dataset(path: Path) -> List[Dict[str, Any]]:
    rows: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"[warn] could not parse line {line_no}: {e}",
                      file=sys.stderr)
    return rows
 
 
def already_done(out_path: Path) -> bool:
    if not out_path.exists():
        return False
    try:
        with out_path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        return bool(data.get("response"))
    except Exception:
        return False
 
 
def atomic_write_json(path: Path, payload: Dict[str, Any]) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with tmp.open("w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)
    tmp.replace(path)
 
 